# Gateway

Local reverse proxy on `PORT` (default 8080): `/musicgen` → MusicGen, `/spleeter` → Spleeter, `/magentart` → Magenta RT Holly. Start those API cells first, then run **the code cell below once** and leave this kernel running.

By default this cell also opens an **ngrok HTTP tunnel** to that local port so you get one public URL for every routed backend. Set `USE_NGROK=0` to skip (local-only). Prefer `NGROK_AUTHTOKEN` in the environment on a headless server; otherwise you will be prompted.

See `.env.example` for `PORT`, upstream URLs, and ngrok-related variables.


In [ ]:
import os
import threading
import time
from contextlib import asynccontextmanager
from typing import AsyncIterator

import httpx
import nest_asyncio
import uvicorn
from fastapi import FastAPI, Request
from fastapi.middleware.cors import CORSMiddleware
from starlette.responses import Response, StreamingResponse

nest_asyncio.apply()

MUSICGEN_UPSTREAM = os.environ.get("MUSICGEN_UPSTREAM", "http://127.0.0.1:8101").rstrip("/")
SPLEETER_UPSTREAM = os.environ.get("SPLEETER_UPSTREAM", "http://127.0.0.1:8102").rstrip("/")
MAGENTART_UPSTREAM = os.environ.get("MAGENTART_UPSTREAM", "http://127.0.0.1:8103").rstrip("/")
GATEWAY_PORT = int(os.environ.get("PORT", "65432"))

REQUEST_DROP = frozenset(
    {
        "connection",
        "keep-alive",
        "proxy-authenticate",
        "proxy-authorization",
        "te",
        "trailers",
        "upgrade",
        "host",
    }
)

RESPONSE_DROP = frozenset({"connection", "keep-alive"})

METHODS = ("GET", "POST", "PUT", "DELETE", "OPTIONS", "PATCH", "HEAD")


@asynccontextmanager
async def lifespan(app: FastAPI):
    timeout = httpx.Timeout(connect=60.0, read=900.0, write=900.0, pool=60.0)
    async with httpx.AsyncClient(timeout=timeout, follow_redirects=False) as client:
        app.state.http_client = client
        yield


app = FastAPI(title="Code of Music ML Gateway", lifespan=lifespan)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)


@app.get("/")
async def root():
    return {
        "message": "Gateway for MusicGen, Spleeter, and Magenta RT Holly",
        "musicgen_prefix": "/musicgen",
        "spleeter_prefix": "/spleeter",
        "magentart_prefix": "/magentart",
    }


def _filter_response_headers(resp: httpx.Response) -> dict[str, str]:
    out: dict[str, str] = {}
    for k, v in resp.headers.items():
        lk = k.lower()
        if lk in RESPONSE_DROP or lk.startswith("access-control-"):
            continue
        out[k] = v
    return out


def _filter_request_headers(request: Request, *, streaming_body: bool) -> dict[str, str]:
    out: dict[str, str] = {}
    for k, v in request.headers.items():
        lk = k.lower()
        if lk in REQUEST_DROP:
            continue
        if streaming_body and lk == "content-length":
            continue
        out[k] = v
    return out


async def _proxy_to_upstream(request: Request, upstream_base: str, subpath: str) -> Response:
    path = subpath.lstrip("/")
    url = f"{upstream_base}/{path}" if path else upstream_base
    q = request.url.query
    if q:
        url = f"{url}?{q}"

    client: httpx.AsyncClient = request.app.state.http_client
    stream_body = request.method not in ("GET", "HEAD")

    headers = _filter_request_headers(request, streaming_body=stream_body)

    if stream_body:

        async def body_iter() -> AsyncIterator[bytes]:
            async for chunk in request.stream():
                yield chunk

        req = client.build_request(
            request.method,
            url,
            headers=headers,
            content=body_iter(),
        )
    else:
        req = client.build_request(request.method, url, headers=headers)

    resp = await client.send(req, stream=True)

    async def stream_out() -> AsyncIterator[bytes]:
        try:
            async for chunk in resp.aiter_raw():
                yield chunk
        finally:
            await resp.aclose()

    return StreamingResponse(
        stream_out(),
        status_code=resp.status_code,
        headers=_filter_response_headers(resp),
    )


@app.api_route("/musicgen", methods=METHODS)
async def proxy_musicgen_root(request: Request):
    return await _proxy_to_upstream(request, MUSICGEN_UPSTREAM, "")


@app.api_route("/musicgen/{path:path}", methods=METHODS)
async def proxy_musicgen(request: Request, path: str):
    return await _proxy_to_upstream(request, MUSICGEN_UPSTREAM, path)


@app.api_route("/spleeter", methods=METHODS)
async def proxy_spleeter_root(request: Request):
    return await _proxy_to_upstream(request, SPLEETER_UPSTREAM, "")


@app.api_route("/spleeter/{path:path}", methods=METHODS)
async def proxy_spleeter(request: Request, path: str):
    return await _proxy_to_upstream(request, SPLEETER_UPSTREAM, path)


@app.api_route("/magentart", methods=METHODS)
async def proxy_magentart_root(request: Request):
    return await _proxy_to_upstream(request, MAGENTART_UPSTREAM, "")


@app.api_route("/magentart/{path:path}", methods=METHODS)
async def proxy_magentart(request: Request, path: str):
    return await _proxy_to_upstream(request, MAGENTART_UPSTREAM, path)


def start_gateway():
    def run():
        print(
            f"Gateway listening http://0.0.0.0:{GATEWAY_PORT} → "
            f"musicgen {MUSICGEN_UPSTREAM} | spleeter {SPLEETER_UPSTREAM} | magentart {MAGENTART_UPSTREAM}"
        )
        uvicorn.run(app, host="0.0.0.0", port=GATEWAY_PORT, log_level="info")

    threading.Thread(target=run, daemon=True).start()
    time.sleep(2)
    print("Gateway thread started")


def maybe_start_ngrok():
    off = os.environ.get("USE_NGROK", "1").lower() in ("0", "false", "no")
    if off:
        print(
            f"ngrok skipped (USE_NGROK=0). Public access: run `ngrok http {GATEWAY_PORT}` "
            "or set USE_NGROK=1."
        )
        return

    from pyngrok import ngrok

    token = (os.environ.get("NGROK_AUTHTOKEN") or "").strip()
    if not token:
        from getpass import getpass

        print("Token: https://dashboard.ngrok.com/get-started/your-authtoken")
        token = getpass("NGROK_AUTHTOKEN (empty to skip ngrok): ").strip()
    if not token:
        print("No ngrok token; only local URL is active.")
        return

    ngrok.set_auth_token(token)
    domain = (os.environ.get("NGROK_DOMAIN") or "").strip()
    if domain:
        public = ngrok.connect(GATEWAY_PORT, domain=domain)
    else:
        public = ngrok.connect(GATEWAY_PORT)
    print("Public URL:", public)
    print("Set p5 YOUR_NGROK_URL to this (useSharedGateway: true).")


start_gateway()
maybe_start_ngrok()
